# Simulação do Protocolo de Árvore Geradora (STP / RSTP) no Sistema SCADA-Core
## Estudo de Caso: Estação de Reabastecimento de Hidrogênio (HRS) — ECAA08

Este notebook simula o protocolo **RSTP (IEEE 802.1w)** aplicado à rede Ethernet industrial da Estação de Reabastecimento de Hidrogênio, validando:
1. Eleição determinística do *Root Bridge* (`SW-CORE`, prioridade `4096`);
2. Cálculo de custos acumulados e atribuição de papéis (*Root Port*, *Designated Port*, *Alternate/Discarding*);
3. Comutação automática (*failover*) após rompimento do *backbone* óptico entre `SW-CORE` e `SW-S100`;
4. Auditoria formal das propriedades matemáticas de Árvore Geradora ($|E_T| = |V| - 1$, conexa e acíclica).

### 1. Importação de Bibliotecas e Modelagem do Grafo da Rede HRS

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

G = nx.Graph()

# Switches gerenciaveis da HRS com Bridge Priority e MAC
switches = {
    "SW-CORE": {"prio": 4096,  "mac": "00:1A:2B:00:00:01", "setor": "Sala de Controle SCADA-01"},
    "SW-S100": {"prio": 8192,  "mac": "00:1A:2B:00:01:00", "setor": "Setor 100 (ELET-01 / COMP-01)"},
    "SW-S200": {"prio": 32768, "mac": "00:1A:2B:00:02:00", "setor": "Setor 200 (TK-AP / RESF-01)"},
    "SW-S300": {"prio": 32768, "mac": "00:1A:2B:00:03:00", "setor": "Setor 300 (DISP-01 Pista)"},
    "SW-SIS":  {"prio": 32768, "mac": "00:1A:2B:00:04:00", "setor": "Painel SIS (ESD-100 / AT-101)"},
}

for sw, attrs in switches.items():
    G.add_node(sw, **attrs)

# Enlaces fisicos redundantes (custos IEEE 802.1w: 10G=2000, 1G=20000, 100M=200000)
enlaces = [
    ("SW-CORE", "SW-S100", 2000,  "10 Gbps Fibra"),
    ("SW-CORE", "SW-SIS",  20000, "1 Gbps Fibra"),
    ("SW-S100", "SW-S200", 20000, "1 Gbps Fibra"),
    ("SW-S200", "SW-S300", 20000, "1 Gbps Fibra"),
    ("SW-S300", "SW-SIS",  20000, "1 Gbps Fibra"),
    ("SW-S100", "SW-SIS",  200000, "100 Mbps Backup"),
]

for u, v, cost, desc in enlaces:
    G.add_edge(u, v, cost=cost, desc=desc)

print(f"Rede fisica HRS modelada: {G.number_of_nodes()} switches e {G.number_of_edges()} enlaces.")

### 2. Implementação do Algoritmo RSTP (Eleição de *Root Bridge*, Árvore Geradora e Papéis de Porta)

In [ ]:
def executar_rstp(grafo: nx.Graph):
    """
    Simula a convergencia IEEE 802.1w (RSTP):
    1. Elege o Root Bridge pelo menor Bridge ID (prio, mac).
    2. Calcula caminhos de menor custo acumulado ate o Root Bridge.
    3. Classifica enlaces em Forwarding (ativos na arvore) e Alternate/Discarding (bloqueados).
    """
    root = min(grafo.nodes, key=lambda n: (grafo.nodes[n]["prio"], grafo.nodes[n]["mac"]))

    # Dijkstra para menor custo ate o Root Bridge
    dist = {n: float("inf") for n in grafo.nodes}
    parent = {n: None for n in grafo.nodes}
    dist[root] = 0
    nao_visitados = set(grafo.nodes)

    while nao_visitados:
        u = min(nao_visitados, key=lambda n: (dist[n], grafo.nodes[n]["prio"], grafo.nodes[n]["mac"]))
        nao_visitados.remove(u)
        if dist[u] == float("inf"):
            break
        for v in grafo.neighbors(u):
            alt = dist[u] + grafo[u][v]["cost"]
            if alt < dist[v]:
                dist[v] = alt
                parent[v] = u
            elif alt == dist[v] and parent[v] is not None:
                # Desempate deterministico por menor Bridge ID do vizinho
                bid_u = (grafo.nodes[u]["prio"], grafo.nodes[u]["mac"])
                bid_p = (grafo.nodes[parent[v]]["prio"], grafo.nodes[parent[v]]["mac"])
                if bid_u < bid_p:
                    parent[v] = u

    arestas_ativas = []
    for n, p in parent.items():
        if p is not None:
            arestas_ativas.append(tuple(sorted((n, p))))

    set_ativas = set(arestas_ativas)
    arestas_bloqueadas = [tuple(sorted((u, v))) for u, v in grafo.edges if tuple(sorted((u, v))) not in set_ativas]

    tabela_portas = []
    for u, v in grafo.edges:
        chave = tuple(sorted((u, v)))
        custo = grafo[u][v]["cost"]
        if chave in set_ativas:
            if parent[v] == u:
                tabela_portas.append({"Enlace": f"{u} <-> {v}", "Custo": custo, "Estado": "Forwarding", f"Papel em {u}": "Designated", f"Papel em {v}": "Root Port"})
            else:
                tabela_portas.append({"Enlace": f"{u} <-> {v}", "Custo": custo, "Estado": "Forwarding", f"Papel em {v}": "Designated", f"Papel em {u}": "Root Port"})
        else:
            tabela_portas.append({"Enlace": f"{u} <-> {v}", "Custo": custo, "Estado": "Discarding (Alternate)", "Detalhe": "Loop prevenido; Hot Standby"})

    return root, dist, arestas_ativas, arestas_bloqueadas, pd.DataFrame(tabela_portas)

root_norm, dist_norm, ativas_norm, bloq_norm, df_norm = executar_rstp(G)
print(f"Root Bridge Eleito: {root_norm} (Priority={G.nodes[root_norm]['prio']})")
print("Custos acumulados ate o Root:", dist_norm)
display(df_norm.fillna("-"))

### 3. Visualização Gráfica: Operação Normal vs. Falha Crítica no Backbone `SW-CORE` $\leftrightarrow$ `SW-S100`

In [ ]:
def plotar_topologia_rstp(ax, grafo, root, ativas, bloqueadas, titulo):
    pos = {
        "SW-CORE": (0.5, 1.0),
        "SW-S100": (0.0, 0.55),
        "SW-SIS":  (1.0, 0.55),
        "SW-S200": (0.15, 0.0),
        "SW-S300": (0.85, 0.0),
    }
    node_colors = ["gold" if n == root else "lightblue" for n in grafo.nodes]
    nx.draw_networkx_nodes(grafo, pos, ax=ax, node_color=node_colors, node_size=1800, edgecolors="black")
    nx.draw_networkx_labels(grafo, pos, ax=ax, font_size=8.5, font_weight="bold")

    nx.draw_networkx_edges(grafo, pos, ax=ax, edgelist=ativas, width=2.8, edge_color="green", label="Forwarding")
    nx.draw_networkx_edges(grafo, pos, ax=ax, edgelist=bloqueadas, width=2.0, style="dashed", edge_color="red", label="Discarding (Alternate)")

    edge_labels = {(u, v): f"{grafo[u][v]['cost']}" for u, v in grafo.edges}
    nx.draw_networkx_edge_labels(grafo, pos, ax=ax, edge_labels=edge_labels, font_size=8)
    ax.set_title(titulo, fontsize=11)
    ax.axis("off")

# Cenario 2: Rompimento do backbone SW-CORE <-> SW-S100
G_falha = G.copy()
G_falha.remove_edge("SW-CORE", "SW-S100")
root_falha, dist_falha, ativas_falha, bloq_falha, df_falha = executar_rstp(G_falha)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plotar_topologia_rstp(axes[0], G, root_norm, ativas_norm, bloq_norm, "Operação Normal (Root: SW-CORE)")
plotar_topologia_rstp(axes[1], G_falha, root_falha, ativas_falha, bloq_falha, "Falha Backbone SW-CORE <-> SW-S100 (Failover RSTP)")
plt.tight_layout()
plt.show()

print("Novos custos apos failover:", dist_falha)

### 4. Auditoria Matemática da Árvore Geradora Resultante

In [ ]:
def auditar_arvore_geradora(grafo_original: nx.Graph, arestas_ativas, nome: str):
    T = nx.Graph()
    T.add_nodes_from(grafo_original.nodes)
    T.add_edges_from(arestas_ativas)
    conexo = nx.is_connected(T)
    aciclico = nx.is_forest(T)
    n_arestas_ok = T.number_of_edges() == grafo_original.number_of_nodes() - 1
    print(f"[{nome}] Conexo={conexo} | Aciclico={aciclico} | |E_T|==|V|-1 ({T.number_of_edges()}=={grafo_original.number_of_nodes()-1}): {n_arestas_ok}")
    assert conexo and aciclico and n_arestas_ok

auditar_arvore_geradora(G, ativas_norm, "Operacao Normal")
auditar_arvore_geradora(G_falha, ativas_falha, "Pos-Failover")
print("Auditoria formal concluida: ambas as topologias sao arvores geradoras validas.")